<a href="https://colab.research.google.com/github/dilanmano20010-design/Teoria-juegos-decisiones-consumidor/blob/main/investigacion_teoria_de_juegos_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Configuración del Entorno e Instalación de Librerías
En Google Colab instalamos librerías cuantitativas para la resolución algorítmica de juegos y visualización interactiva avanzada:
* `nashpy`: Resolución analítica y numérica de equilibrios de Nash.
* `plotly`: Gráficos interactivos de alta definición y motores 3D basados en WebGL.
* `scipy`: Integración de sistemas de ecuaciones diferenciales para dinámica replicadora.


#  Teoría de Juegos y Economía Estratégica
### Análisis Cuantitativo, Modelos de Oligopolio, Dinámica Evolutiva y Modelado en 3D

---
* **Autor / Investigador**: **Dilan Alexander Manosalvas Andrade**
* **Sitio Web Oficial / Portafolio**: [https://econometricis.vercel.app/](https://econometricis.vercel.app/)
* **Plataforma de Ejecución**: Google Colab / Python para Microeconomía, Econometría y Teoría de Juegos
* **Contenido Principal**:
  * Fundamentos de Interacción Estratégica (Forma Normal y Extensiva)
  * Equilibrios de Nash Puros y Mixtos (Dilema del Prisionero, Halcón-Paloma, Batalla de los Sexos)
  * Organización Industrial: Duopolio de Cournot, Paradoja de Bertrand y Estabilidad de Cárteles
  * Gráfico Interactivo de Funciones de Mejor Respuesta (Plotly)
  * ** Modelo 3D**: Superficie Tridimensional de Beneficios de Cournot y Equilibrio de Nash (`go.Surface`)
  * Juegos Repetidos y Torneo de Estrategias Dinámicas (Tit-for-Tat, Grim Trigger, Always Defect)
  * ** Modelo 3D**: Dinámica Replicadora Evolutiva en el Espacio de Fases Tridimensional (`go.Scatter3d`)
  * Aplicaciones del Mundo Real: Guerras de Precios, OPEP y Ecosistemas Tecnológicos
  * Conclusiones Cuantitativas, Q&A y Próximos Pasos


In [ ]:
# Instalación automática en Google Colab
!pip install -q nashpy plotly scipy statsmodels sympy


## 2. Importación de Módulos y Parámetros Estéticos


In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.integrate import odeint
import nashpy as nash

# Configuración estética
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✅ Entorno de Teoría de Juegos configurado con éxito.")
print("Versión de Python:", sys.version.split()[0])
print("Versión de Numpy:", np.__version__)
print("Versión de Nashpy:", nash.__version__)


✅ Entorno de Teoría de Juegos configurado con éxito.
Versión de Python: 3.13.15
Versión de Numpy: 2.1.3
Versión de Nashpy: 0.0.43


## 3. Fundamentos de Juegos Simultáneos 2x2 y Equilibrio de Nash
Un juego simultáneo en forma normal se define por $\Gamma = \langle N, (S_i)_{i \in N}, (u_i)_{i \in N} \rangle$:
* $N = \{1, 2, \dots, n\}$: Conjunto finito de jugadores racionales.
* $S_i$: Conjunto de estrategias disponibles para el jugador $i$.
* $u_i(s_1, \dots, s_n)$: Función de pagos o utilidad que mapea el perfil de estrategias en $\mathbb{R}$.

### El Dilema del Prisionero
Dos sospechosos arrestados pueden **Cooperar (C)** guardando silencio o **Defraudar (D)** confesando.
* Matriz de pagos tradicional:
  * Si ambos cooperan: $(-1, -1)$
  * Si uno defrauda y el otro coopera: $(0, -3)$
  * Si ambos defraudan: $(-2, -2)$

El perfil $(D, D)$ es el **único Equilibrio de Nash en estrategias estrictamente dominantes**, pero resulta **ineficiente en el sentido de Pareto** frente al resultado colectivo superior $(C, C)$.


In [ ]:
A_pd = np.array([[-1, -3],
                 [ 0, -2]])
B_pd = np.array([[-1,  0],
                 [-3, -2]])

prisoner_dilemma = nash.Game(A_pd, B_pd)
print("Matriz de Pagos - Dilema del Prisionero:")
print("Jugador 1 (Fila):\n", A_pd)
print("Jugador 2 (Columna):\n", B_pd)

equilibria = list(prisoner_dilemma.support_enumeration())
print("\nEquilibrios de Nash encontrados:")
for eq in equilibria:
    p1_strat, p2_strat = eq
    print(f"  • Jugador 1: Probabilidad [Cooperar, Defraudar] = {p1_strat}")
    print(f"  • Jugador 2: Probabilidad [Cooperar, Defraudar] = {p2_strat}")

Matriz de Pagos - Dilema del Prisionero:
Jugador 1 (Fila):
 [[-1 -3]
 [ 0 -2]]
Jugador 2 (Columna):
 [[-1  0]
 [-3 -2]]

Equilibrios de Nash encontrados:
  • Jugador 1: Probabilidad [Cooperar, Defraudar] = [0. 1.]
  • Jugador 2: Probabilidad [Cooperar, Defraudar] = [0. 1.]


## 4. Juegos de Conflicto y Estrategias Mixtas: Halcón-Paloma (*Hawk-Dove*)
Cuando existen múltiples equilibrios o tensiones competitivas, los jugadores asignan probabilidades a sus decisiones:
$$\sigma_i \in \Delta(S_i) \quad \text{con} \quad \sum_{s \in S_i} \sigma_i(s) = 1$$

En el juego de **Halcón-Paloma**, dos agentes compiten por un recurso valioso $V$ con un costo de combate o daño $C$ ($C > V$):
* **Halcón vs. Halcón**: Enfrentamiento violento con pago esperado $\frac{V-C}{2}$.
* **Halcón vs. Paloma**: El halcón se apodera del recurso completo ($V, 0$).
* **Paloma vs. Paloma**: Comparten el recurso pacíficamente ($\frac{V}{2}, \frac{V}{2}$).


In [ ]:
V = 4 # Valor del recurso en disputa
C = 6 # Costo de herida / combate

A_hd = np.array([[(V - C)/2, V],
                 [0,         V/2]])
B_hd = np.array([[(V - C)/2, 0],
                 [V,         V/2]])

hawk_dove = nash.Game(A_hd, B_hd)
print(f"Juego Halcón-Paloma con Valor V={V} y Costo de Conflicto C={C}:")
display(pd.DataFrame(A_hd, index=['Halcón', 'Paloma'], columns=['Halcón', 'Paloma']))

hd_equilibria = list(hawk_dove.support_enumeration())
print("\nEquilibrios de Nash:")
for i, eq in enumerate(hd_equilibria, 1):
    tipo = "Estrategias Puras" if np.all(np.isin(eq[0], [0, 1])) else "Estrategia Mixta"
    print(f"  • Equilibrio {i} ({tipo}): Jugador 1={eq[0]} | Jugador 2={eq[1]}")

Juego Halcón-Paloma con Valor V=4 y Costo de Conflicto C=6:


,Halcón,Paloma
Halcón,-1.0,4.0
Paloma,0.0,2.0



Equilibrios de Nash:
  • Equilibrio 1 (Estrategias Puras): Jugador 1=[1. 0.] | Jugador 2=[0. 1.]
  • Equilibrio 2 (Estrategias Puras): Jugador 1=[0. 1.] | Jugador 2=[1. 0.]
  • Equilibrio 3 (Estrategia Mixta): Jugador 1=[0.66666667 0.33333333] | Jugador 2=[0.66666667 0.33333333]


## 5. Organización Industrial: Duopolio de Cournot y Competencia Estratégica
En un mercado oligopólico donde dos empresas compiten fijando simultáneamente cantidades $q_1$ y $q_2$:
* Demanda inversa lineal: $P(Q) = a - b Q = a - b(q_1 + q_2)$
* Costo marginal constante: $c$ ($a > c > 0$)
* Beneficio de la firma 1: $\pi_1(q_1, q_2) = (a - b(q_1 + q_2) - c) q_1$

### Derivación de la Función de Mejor Respuesta (*Best Response*):
$$\frac{\partial \pi_1}{\partial q_1} = a - c - 2b q_1 - b q_2 = 0 \implies q_1^*(q_2) = \frac{a - c - b q_2}{2b}$$
Por simetría:
$$q_2^*(q_1) = \frac{a - c - b q_1}{2b}$$

En el **Equilibrio de Nash de Cournot**:
$$q_1^N = q_2^N = \frac{a - c}{3b}, \quad Q^N = \frac{2(a - c)}{3b}, \quad P^N = \frac{a + 2c}{3}$$

Frente al **Acuerdo de Colusión / Cártel (Monopolio Compartido)**:
$$q_1^C = q_2^C = \frac{a - c}{4b}, \quad Q^C = \frac{a - c}{2b}, \quad \pi_i^C > \pi_i^N$$


In [ ]:
# Parámetros macroeconómicos del mercado
a = 120   # Precio de reserva
b = 1.0   # Sensibilidad al precio
c = 30    # Costo marginal unitario

# Soluciones analíticas
q_nash = (a - c) / (3 * b)
q_cartel = (a - c) / (4 * b)
q_comp = (a - c) / (2 * b) # Competencia perfecta

pi_nash = (a - b * (2 * q_nash) - c) * q_nash
pi_cartel = (a - b * (2 * q_cartel) - c) * q_cartel

print(" Soluciones Analíticas del Duopolio:")
print(f"  • Equilibrio de Nash (Cournot): q1 = q2 = {q_nash:.2f} | Beneficio unitario: ${pi_nash:.2f}")
print(f"  • Colusión / Cártel:            q1 = q2 = {q_cartel:.2f} | Beneficio unitario: ${pi_cartel:.2f}")
print(f"  • Competencia Perfecta (P=c):   Q total = {(a-c)/b:.2f} | Beneficio económico: $0.00")


 Soluciones Analíticas del Duopolio:
  • Equilibrio de Nash (Cournot): q1 = q2 = 30.00 | Beneficio unitario: $900.00
  • Colusión / Cártel:            q1 = q2 = 22.50 | Beneficio unitario: $1012.50
  • Competencia Perfecta (P=c):   Q total = 90.00 | Beneficio económico: $0.00


## 6. Curvas de Reacción Interactivas de Cournot con Plotly
Visualizamos el cruce de las funciones de mejor respuesta donde se intersectan en el punto de Nash, comparándolo con la cuota cooperativa de colusión.


In [ ]:
q_range = np.linspace(0, (a - c) / b, 200)

br1 = np.clip((a - c - b * q_range) / (2 * b), 0, None)
br2 = np.clip((a - c - b * q_range) / (2 * b), 0, None)

fig_br = go.Figure()

# Mejor respuesta Firma 1
fig_br.add_trace(go.Scatter(
    x=q_range, y=br1,
    mode='lines',
    name='Mejor Respuesta Firma 1: q1*(q2)',
    line=dict(color='#3b82f6', width=3)
))

# Mejor respuesta Firma 2
fig_br.add_trace(go.Scatter(
    x=br2, y=q_range,
    mode='lines',
    name='Mejor Respuesta Firma 2: q2*(q1)',
    line=dict(color='#ef4444', width=3)
))

# Punto de Nash
fig_br.add_trace(go.Scatter(
    x=[q_nash], y=[q_nash],
    mode='markers+text',
    name=f'Equilibrio de Nash ({q_nash:.1f}, {q_nash:.1f})',
    marker=dict(size=14, color='#10b981', symbol='diamond'),
    text=[' Nash (Cournot)'],
    textposition='top right'
))

# Punto de Colusión
fig_br.add_trace(go.Scatter(
    x=[q_cartel], y=[q_cartel],
    mode='markers+text',
    name=f'Colusión / Cártel ({q_cartel:.1f}, {q_cartel:.1f})',
    marker=dict(size=14, color='#f59e0b', symbol='star'),
    text=[' Colusión (Cártel)'],
    textposition='bottom left'
))

fig_br.update_layout(
    title='<b>Funciones de Mejor Respuesta en el Duopolio de Cournot</b>',
    xaxis_title='Cantidad de la Firma 1 (q1)',
    yaxis_title='Cantidad de la Firma 2 (q2)',
    template='plotly_dark',
    height=650,
    xaxis=dict(range=[0, (a - c) / b]),
    yaxis=dict(range=[0, (a - c) / b])
)

fig_br.show()


## 7.  Visualización 3D: Superficie del Paisaje de Beneficios de Cournot
En el espacio tridimensional $(q_1, q_2, \pi_1)$, la función de beneficio de una firma forma una colina convexa:
* Eje X: Cantidad de la Firma 1 ($q_1$)
* Eje Y: Cantidad de la Firma 2 ($q_2$)
* Eje Z: Beneficio Económico de la Firma 1 ($\pi_1$)

Comparamos en 3D:
1.  **Equilibrio de Nash**: Estable pero subóptimo.
2.  **Cártel / Colusión**: Máximo bienestar conjunto.
3.  **Incentivo a Traicionar / Romper el Cártel**: Si la firma rival produce la cuota pactada $q_2^C$, la firma 1 maximiza beneficios aumentando su producción unilateralmente, lo que explica la fragilidad intrínseca de cárteles como la OPEP.

 *Haz clic y arrastra con el ratón para rotar en 360 grados, hacer zoom y explorar la superficie en 3D.*


In [ ]:
# Malla tridimensional de cantidades
q1_mesh = np.linspace(5, (a - c) / b, 60)
q2_mesh = np.linspace(5, (a - c) / b, 60)
Q1, Q2 = np.meshgrid(q1_mesh, q2_mesh)

Price_mesh = a - b * (Q1 + Q2)
Pi1_mesh = (Price_mesh - c) * Q1

fig_cournot_3d = go.Figure()

# Superficie 3D
fig_cournot_3d.add_trace(go.Surface(
    x=q1_mesh,
    y=q2_mesh,
    z=Pi1_mesh,
    colorscale='Viridis',
    colorbar=dict(title='Beneficio Firma 1 ($)', thickness=15),
    opacity=0.88,
    name='Beneficio Pi1(q1, q2)'
))

# Marcador 3D Nash
fig_cournot_3d.add_trace(go.Scatter3d(
    x=[q_nash], y=[q_nash], z=[pi_nash],
    mode='markers+text',
    marker=dict(size=9, color='lime', symbol='diamond'),
    text=['⭐ Equilibrio de Nash'],
    textposition='top center',
    name=f'Nash: Pi1=${pi_nash:.1f}'
))

# Marcador 3D Cártel
fig_cournot_3d.add_trace(go.Scatter3d(
    x=[q_cartel], y=[q_cartel], z=[pi_cartel],
    mode='markers+text',
    marker=dict(size=9, color='gold', symbol='circle'),
    text=['👑 Colusión (Cártel)'],
    textposition='top center',
    name=f'Cártel: Pi1=${pi_cartel:.1f}'
))

# Marcador 3D Desviación Unilateral
q_cheat = (a - c - b * q_cartel) / (2 * b)
pi_cheat = (a - b * (q_cheat + q_cartel) - c) * q_cheat

fig_cournot_3d.add_trace(go.Scatter3d(
    x=[q_cheat], y=[q_cartel], z=[pi_cheat],
    mode='markers+text',
    marker=dict(size=9, color='red', symbol='cross'),
    text=['🚨 Traición / Desviación'],
    textposition='top center',
    name=f'Trampa Cártel: Pi1=${pi_cheat:.1f}'
))

fig_cournot_3d.update_layout(
    title='<b>Paisaje Tridimensional de Beneficios de Cournot en 3D</b><br><sup>(Arrastra con el ratón para rotar 360° y visualizar el incentivo a romper cárteles)</sup>',
    scene=dict(
        xaxis_title='Cantidad Firma 1 (q1)',
        yaxis_title='Cantidad Firma 2 (q2)',
        zaxis_title='Beneficio Firma 1 ($)',
        xaxis=dict(backgroundcolor="#111827", gridcolor="#374151"),
        yaxis=dict(backgroundcolor="#111827", gridcolor="#374151"),
        zaxis=dict(backgroundcolor="#111827", gridcolor="#374151"),
        camera=dict(eye=dict(x=-1.6, y=-1.6, z=1.1))
    ),
    template='plotly_dark',
    height=800,
    margin=dict(l=0, r=0, b=0, t=50)
)

fig_cournot_3d.show()

print(f"Incentivo a desviar del cártel: Si la Firma 2 respeta la cuota ({q_cartel:.1f}), la Firma 1 gana ${pi_cheat:.1f} produciendo {q_cheat:.1f} en lugar de ${pi_cartel:.1f}.")


Incentivo a desviar del cártel: Si la Firma 2 respeta la cuota (22.5), la Firma 1 gana $1139.1 produciendo 33.8 en lugar de $1012.5.


## 8. Juegos Repetidos y Torneo Dinámico de Estrategias (Robert Axelrod)
En interacciones continuas de mercado, el **Teorema del Folklore** demuestra que la cooperación puede sostenerse si el factor de descuento $\delta$ es suficientemente alto.
Simulamos un torneo dinámico de 60 rondas comparando 4 estrategias paradigmáticas:
* **Tit-for-Tat (Toma y Daca)**: Comienza cooperando y en cada ronda sucesiva replica la jugada anterior de su oponente.
* **Grim Trigger (Gatillo Sombrío)**: Coopera hasta que el rival defrauda una sola vez; a partir de ahí castiga defraudando permanentemente.
* **Always Defect (Siempre Traicionar)**: Defrauda incondicionalmente en todas las rondas.
* **Random (Aleatoria)**: Elige al azar con 50% de probabilidad.


In [ ]:
rounds = 60

def play_tournament():
    payoffs_matrix = np.array([
        [(3, 3), (0, 5)],  # [C, C], [C, D]
        [(5, 0), (1, 1)]   # [D, C], [D, D]
    ])

    strategies = ['Tit-for-Tat', 'Grim Trigger', 'Always Defect', 'Random']
    history_scores = {s: np.zeros(rounds) for s in strategies}

    for i, s1 in enumerate(strategies):
        for j, s2 in enumerate(strategies):
            if i >= j: continue

            s1_moves, s2_moves = [], []
            grim_triggered_1, grim_triggered_2 = False, False

            for t in range(rounds):
                # Decisión s1
                if s1 == 'Tit-for-Tat':
                    m1 = 0 if t == 0 else s2_moves[-1]
                elif s1 == 'Grim Trigger':
                    m1 = 1 if grim_triggered_1 else 0
                elif s1 == 'Always Defect':
                    m1 = 1
                else:
                    m1 = np.random.choice([0, 1])

                # Decisión s2
                if s2 == 'Tit-for-Tat':
                    m2 = 0 if t == 0 else s1_moves[-1]
                elif s2 == 'Grim Trigger':
                    m2 = 1 if grim_triggered_2 else 0
                elif s2 == 'Always Defect':
                    m2 = 1
                else:
                    m2 = np.random.choice([0, 1])

                if m2 == 1: grim_triggered_1 = True
                if m1 == 1: grim_triggered_2 = True

                s1_moves.append(m1)
                s2_moves.append(m2)

                history_scores[s1][t] += payoffs_matrix[m1, m2][0]
                history_scores[s2][t] += payoffs_matrix[m1, m2][1]

    return history_scores

scores = play_tournament()

fig_axelrod = go.Figure()
for strat, score_arr in scores.items():
    cum_score = np.cumsum(score_arr)
    fig_axelrod.add_trace(go.Scatter(
        x=list(range(1, rounds + 1)),
        y=cum_score,
        mode='lines+markers',
        name=strat,
        line=dict(width=2.5)
    ))

fig_axelrod.update_layout(
    title='<b>Torneo de Estrategias Repetidas: Acumulación Temporal de Pagos</b>',
    xaxis_title='Número de Ronda',
    yaxis_title='Puntaje Acumulado',
    template='plotly_dark',
    height=600,
    hovermode='x unified'
)

fig_axelrod.show()


## 9.  Teoría de Juegos Evolutiva y Dinámica Replicadora en el Espacio 3D
En la **Teoría de Juegos Evolutiva**, los agentes económicos no poseen clarividencia; las estrategias con mayor rendimiento se propagan por imitación y selección de mercado según la **Ecuación Replicadora**:
$$\dot{x}_i = x_i \cdot \left[ (A x)_i - x^T A x \right]$$
Donde:
* $x_i$: Participación de mercado o proporción poblacional que adopta la estrategia $i$.
* $(A x)_i$: Aptitud (*fitness*) esperada de la estrategia $i$.
* $x^T A x$: Aptitud promedio de todo el ecosistema de mercado.

Simulamos un sistema tridimensional de 3 estrategias competitivas cíclicas (p. ej. Estrategia Agresiva, Defensiva e Innovadora).

 *Rotación 3D: Puedes girar la trayectoria orbital tridimensional para observar los ciclos límite y el equilibrio interior.*


In [ ]:
# Matriz de pagos de competencia cíclica entre 3 modelos estratégicos
A_evol = np.array([
    [ 0,  -1,   1.2],
    [ 1,   0,  -1.0],
    [-1.2, 1,   0.0]
])

def replicator_system(x, t, A):
    x = np.clip(x, 1e-7, None)
    x = x / np.sum(x)
    fitness = np.dot(A, x)
    avg_fitness = np.dot(x, fitness)
    dxdt = x * (fitness - avg_fitness)
    return dxdt

t_eval = np.linspace(0, 45, 2500)

initial_conditions = [
    [0.6, 0.2, 0.2],
    [0.2, 0.6, 0.2],
    [0.2, 0.2, 0.6],
    [0.4, 0.3, 0.3],
    [0.45, 0.45, 0.1]
]

fig_replicator_3d = go.Figure()
colors = ['#3b82f6', '#ef4444', '#10b981', '#f59e0b', '#8b5cf6']

for idx, x0 in enumerate(initial_conditions):
    sol = odeint(replicator_system, x0, t_eval, args=(A_evol,))

    fig_replicator_3d.add_trace(go.Scatter3d(
        x=sol[:, 0],
        y=sol[:, 1],
        z=sol[:, 2],
        mode='lines',
        line=dict(color=colors[idx], width=3.5),
        name=f'Trayectoria x0={x0}'
    ))

    fig_replicator_3d.add_trace(go.Scatter3d(
        x=[x0[0]], y=[x0[1]], z=[x0[2]],
        mode='markers',
        marker=dict(size=5, color=colors[idx]),
        showlegend=False
    ))

# Equilibrio interior de Nash
fig_replicator_3d.add_trace(go.Scatter3d(
    x=[1/3], y=[1/3], z=[1/3],
    mode='markers+text',
    marker=dict(size=9, color='white', symbol='cross'),
    text=['Centro de Equilibrio (1/3, 1/3, 1/3)'],
    textposition='top center',
    name='Equilibrio Interior'
))

fig_replicator_3d.update_layout(
    title='<b>Dinámica Replicadora Evolutiva en el Espacio de Fases 3D</b><br><sup>(Ciclos límite y estabilidad de selección estratégica en el mercado)</sup>',
    scene=dict(
        xaxis_title='Estrategia 1 (Agresiva: x1)',
        yaxis_title='Estrategia 2 (Defensiva: x2)',
        zaxis_title='Estrategia 3 (Cooperativa: x3)',
        xaxis=dict(range=[0, 1], backgroundcolor="#111827", gridcolor="#374151"),
        yaxis=dict(range=[0, 1], backgroundcolor="#111827", gridcolor="#374151"),
        zaxis=dict(range=[0, 1], backgroundcolor="#111827", gridcolor="#374151"),
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
    ),
    template='plotly_dark',
    height=800,
    margin=dict(l=0, r=0, b=0, t=50)
)

fig_replicator_3d.show()


## 10. Aplicaciones Económicas al Mundo Real
1. **Inestabilidad Estructural de la OPEP+**:
   La superficie 3D de Cournot demuestra matemáticamente que en todo cártel existe un incentivo positivo individual a bombear barriles adicionales por encima de la cuota acordada si los demás países miembros respetan su cuota. Esto explica por qué los acuerdos de producción colapsan periódicamente requiriendo guerras de precios disciplinarias.
2. **Guerras de Ecosistemas e Inteligencia Artificial**:
   La competencia entre sistemas operativos (Android vs. iOS) y entre arquitecturas de IA (modelos propietarios cerrados de OpenAI/Google vs. modelos abiertos como LLaMA de Meta) opera como un juego de coordinación con externalidades de red. El equilibrio se consolida como punto focal una vez que un jugador supera el umbral crítico de adopción.
3. **Subastas y Teoría de Diseño de Mecanismos**:
   En subastas de espectro electromagnético 5G y publicidad digital en tiempo real (Google AdSense), el diseño de subastas de segundo precio (*Vickrey Auctions*) garantiza la revelación veraz de las valoraciones como estrategia dominante.


## 11. Conclusiones Cuantitativas, Q&A y Hallazgos Clave

### Q&A
* **¿Por qué el Equilibrio de Nash no maximiza el bienestar colectivo?**
  Porque el equilibrio de Nash responde a incentivos individuales no coordinados. En el duopolio y en el dilema del prisionero, la búsqueda racional del beneficio propio conduce a un equilibrio subóptimo de Pareto. Para alcanzar y mantener la colusión socialmente óptima se requiere interacción repetida con mecanismos creíbles de castigo.
* **¿Qué estrategia resulta superior en horizontes temporales prolongados?**
  El torneo empírico de Axelrod ratifica que estrategias benévolas, pero con capacidad inmediata de represalia ante el engaño y disposición al perdón rápido (como Tit-for-Tat), acumulan mayor utilidad acumulada que estrategias puramente depredadoras.
* **¿Qué valor analítico aporta el modelado en 3D?**
  La superficie 3D de beneficios permite visualizar con precisión la colina de incentivos y el "abismo" de pérdidas cuando la competencia se intensifica, mientras que el espacio de fases 3D ilustra que la competencia estratégica evolutiva puede generar oscilaciones persistentes y coexistencia de modelos de negocio en lugar de un único monopolio permanente.

### Data Analysis Key Findings
* **Fragilidad Cuantitativa de Cárteles**: El beneficio de desviación unilateral siempre supera al pago del cártel en un juego estático, evidenciando que los acuerdos comerciales sin supervisión son insostenibles por naturaleza.
* **Trayectorias Orbitales en Dinámica Replicadora**: Los sistemas con asimetría cíclica no colapsan a los vértices puros del simplex, sino que orbitan alrededor del equilibrio interior, garantizando diversidad de agentes en el mercado.

### Insights or Next Steps
* **Juegos con Información Incompleta (Harsanyi / Bayesiano)**: Modelar incertidumbre donde las firmas no conocen con certeza las funciones de costo de sus rivales.
* **Modelos Estocásticos de Filtrado y Aprendizaje por Refuerzo**: Implementar agentes de Machine Learning (Q-Learning) que aprendan a cooperar o competir mediante interacción reiterada.

---
###  Créditos y Autoría
* **Investigador / Autor**: **Dilan Alexander Manosalvas Andrade**
* **Sitio Web Personal y Proyectos**: [econometricis.vercel.app](https://econometricis.vercel.app/)
* *Desarrollado y optimizado para análisis cuantitativo en Google Colab.*
